# Phase 0 — Setup & Schema

Clones the `stance` package, installs it, creates the schema in Neon, and seeds the watchlist + fact aliases.

**Before running:** add Kaggle Secrets `NEON_DATABASE_URL`, `GROQ_API_KEY`, `GOOGLE_AI_API_KEY`, and enable internet on this notebook (Settings > Internet > On).

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/<your-username>/stance.git"  # TODO: set this

if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.config import settings
from stance.db import get_engine, init_schema
from stance.seed import seed_all

assert settings.neon_database_url, "NEON_DATABASE_URL not loaded — check Kaggle Secrets are attached to this notebook."

engine = get_engine()
init_schema(engine)
print("Schema created (idempotent).")

In [ ]:
seed_all(engine)

from sqlalchemy import text
with engine.connect() as conn:
    n_companies = conn.execute(text("SELECT COUNT(*) FROM companies")).scalar()
    n_aliases = conn.execute(text("SELECT COUNT(*) FROM fact_aliases")).scalar()
print(f"companies={n_companies}, fact_aliases={n_aliases}")
assert n_companies == 5
assert n_aliases >= 9

## Exit criteria check

Re-run this whole notebook top-to-bottom a second time — table counts must stay identical (idempotent schema + seed).

In [ ]:
with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables WHERE table_schema='public' ORDER BY 1"
    )).fetchall()
print([t[0] for t in tables])